In [7]:
# 1) Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')

# 2) Load CSV
path = Path('car_prices.csv')  # ensure the file is in your working directory
df = pd.read_csv(path)

# 3) Normalize column names
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

# Inspect
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   year          558837 non-null  int64  
 1   make          548536 non-null  str    
 2   model         548438 non-null  str    
 3   trim          548186 non-null  str    
 4   body          545642 non-null  str    
 5   transmission  493485 non-null  str    
 6   vin           558833 non-null  str    
 7   state         558837 non-null  str    
 8   condition     547017 non-null  float64
 9   odometer      558743 non-null  float64
 10  color         558088 non-null  str    
 11  interior      558088 non-null  str    
 12  seller        558837 non-null  str    
 13  mmr           558799 non-null  float64
 14  sellingprice  558825 non-null  float64
 15  saledate      558825 non-null  str    
dtypes: float64(4), int64(1), str(11)
memory usage: 68.2 MB


In [10]:
# Identify key columns
price_col = 'sellingprice' if 'sellingprice' in df.columns else ('price' if 'price' in df.columns else None)
odo_col   = 'odometer' if 'odometer' in df.columns else ('kilometres' if 'kilometres' in df.columns else None)

if price_col is None:
    raise ValueError("Couldn't find target price column. Expected 'sellingprice' or 'price'.")

# Coerce price and odometer to numeric
for c in [price_col, odo_col]:
    if c in df.columns and df[c].dtype == 'O':
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Convert saledate to datetime if present
if 'saledate' in df.columns:
    # Parse to timezone-aware UTC datetimes
    df['saledate'] = pd.to_datetime(df['saledate'], errors='coerce', utc=True)

# If you don't need timezone-awareness, convert to local-naive timestamps
df['saledate'] = df['saledate'].dt.tz_convert('UTC').dt.tz_localize(None)

# Remove rows with missing target
df = df.dropna(subset=[price_col]).copy()

print("Rows after dropping missing target:", len(df))

/tmp/ipykernel_8028/433280091.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['saledate'] = pd.to_datetime(df['saledate'], errors='coerce', utc=True)


Rows after dropping missing target: 558825
